# Benchmark DB Validation

Use this notebook as the first checkpoint before any charting or paper-facing analysis. It verifies that the benchmark database is reachable, that the expected tables and views exist, and that recent runs produced rows in the raw and aggregate metric tables.

The connection bootstrap is environment-aware. It will try `BENCH_DB_URL` first, then a local Postgres URL on `localhost`, and then the Docker Compose service hostname `db`. That makes the notebook usable both from a local editor session and from the profiler or performance-monitoring service container.

## Stage Map

- **Stage 1: Serialization / transformation**
  Starts immediately before the first field write into the destination representation and ends when the payload is sealed and ready for transport.
- **Stage 2: Transport / wire movement**
  Starts at the send API call and ends when the transport confirms completion.
- **Stage 3: Query / traversal**
  Starts at the first parser or reader call on received bytes and ends when the target value has been extracted.
- **Stage 3 materialize**
  Measures eager reconstruction of native structs from the received representation when a consumer explicitly requires materialized objects.

The current local smoke runs focus on Stage 1 and Stage 3 query metrics for the `fastfhir` and `json_fhir` arms.

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


def build_db_url(host):
    user = os.getenv('POSTGRES_USER', 'bench')
    password = os.getenv('POSTGRES_PASSWORD', 'bench')
    database = os.getenv('POSTGRES_DB', 'benchmark')
    port = os.getenv('POSTGRES_PORT', '5432')
    return f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'


def candidate_db_urls():
    configured = os.getenv('BENCH_DB_URL', '').strip()
    candidates = [
        configured,
        build_db_url('localhost'),
        build_db_url('db'),
    ]

    seen = set()
    ordered = []
    for url in candidates:
        if url and url not in seen:
            ordered.append(url)
            seen.add(url)
    return ordered


def connect_engine():
    errors = []
    for url in candidate_db_urls():
        engine = create_engine(url, future=True, pool_pre_ping=True)
        try:
            with engine.connect() as conn:
                conn.execute(text('SELECT 1'))
            return engine, url
        except Exception as exc:
            errors.append(f'{url} -> {exc}')
            engine.dispose()

    raise RuntimeError('Unable to connect to benchmark database. Tried:\n' + '\n'.join(errors))


def query_frame(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or {})


engine, DB_URL = connect_engine()
print('Using DB URL:', DB_URL)

In [ ]:
catalog_df = query_frame(
    """
    SELECT
        table_name AS object_name,
        table_type AS object_type
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY object_name
    """
)

latest_status_df = query_frame(
    """
    SELECT run_id, environment_name, dataset_version, created_at, last_metric_ts, metric_rows, aggregate_rows
    FROM v_latest_run_status
    ORDER BY created_at DESC, run_id DESC
    LIMIT 10
    """
)

print('Catalog objects found:', len(catalog_df))
display(catalog_df)

print('Most recent run status rows:')
display(latest_status_df)

In [ ]:
required_objects = {
    'manifest_table',
    'raw_metrics_table',
    'aggregate_metrics_table',
    'v_latest_run_status',
    'v_stage_latency_summary',
    'v_time_memory_frontier',
}

found_objects = set(catalog_df['object_name'].tolist()) if not catalog_df.empty else set()
missing_objects = sorted(required_objects - found_objects)

if missing_objects:
    raise AssertionError(f'Missing expected benchmark objects: {missing_objects}')

print('All required benchmark tables and views are present.')

In [ ]:
stage_counts_df = query_frame(
    """
    SELECT
        arm,
        stage,
        COUNT(*) AS row_count,
        ROUND(MIN(duration_us) / 1000.0, 3) AS min_ms,
        ROUND(MAX(duration_us) / 1000.0, 3) AS max_ms
    FROM raw_metrics_table
    GROUP BY arm, stage
    ORDER BY arm, stage
    """
)

print('Per-arm and per-stage raw metric counts:')
display(stage_counts_df)

In [ ]:
recent_runs_df = query_frame(
    """
    SELECT run_id, environment_name, benchmark_commit_sha, created_at
    FROM manifest_table
    ORDER BY created_at DESC, run_id DESC
    LIMIT 10
    """
)

aggregate_preview_df = query_frame(
    """
    SELECT run_id, arm, stage, n_samples, p50_ms, p95_ms, p99_ms, throughput_rps, peak_rss_mb
    FROM aggregate_metrics_table
    ORDER BY id DESC
    LIMIT 20
    """
)

print('Recent manifest rows:')
display(recent_runs_df)

print('Recent aggregate rows:')
display(aggregate_preview_df)

## Interpretation Notes

Use this notebook to answer three questions before deeper analysis:

1. Can the notebook connect to the benchmark database?
2. Did migrations create the expected tables and views?
3. Did the most recent runs persist rows for the stages you intended to measure?

For the current local harness, it is normal to see Stage 1 and Stage 3 query rows without Stage 2 transport or Stage 3 materialization rows. Those later stages should appear only after the corresponding benchmark arms are implemented and persisted.